In [1]:
using LowLevelFEM

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07](cache misses: include_dependency fsize change (2), wrong dep version loaded (1), incompatible header (7))
[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (cache misses: include_dependency fsize change (4), wrong dep version loaded (2), incompatible header (14))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
┌ Info: Skipping precompilation due to precompilable error. Importing LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.
┌ Warning: Replacing docs for `LowLevelFEM.mpcReducedBCData :: Tuple{Any, Any, AbstractMatrix}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:249

SYSTEM: caught 

In [2]:
openGeometry("mpc-1.geo")

#openPreProcessor()

In [3]:
mat1 = Material("body")
mat2 = Material("remote")

U = Field([mat1, mat2], type=:VectorField, dim=2, fieldName=:u, rhsName=:f)
#Φ = Field([mat1, mat2], type=:ScalarField, dim=2, fieldName=:φ, rhsName=:m)

Problem("mpc-1", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false)

In [4]:
Ku = ∫(SymGrad(U) ⋅ [2 1 0; 1 2 0; 0 0 1] ⋅ SymGrad(U), Ω="body")
Ku[:,:]

10×10 SparseArrays.SparseMatrixCSC{Float64, Int64} with 62 stored entries:
  1.0           0.5          -0.5          …  -1.38778e-17   ⋅    ⋅ 
  0.5           1.0          -3.46945e-18     -0.5           ⋅    ⋅ 
 -0.5          -3.46945e-18   1.0              0.5           ⋅    ⋅ 
  1.04083e-17   5.55112e-17  -0.5             -0.5           ⋅    ⋅ 
 -0.5          -0.5           5.55112e-17       ⋅            ⋅    ⋅ 
 -0.5          -0.5           1.38778e-17  …   5.55112e-17   ⋅    ⋅ 
  5.55112e-17   1.38778e-17  -0.5             -0.5           ⋅    ⋅ 
 -1.38778e-17  -0.5           0.5              1.0           ⋅    ⋅ 
   ⋅             ⋅             ⋅                ⋅            ⋅    ⋅ 
   ⋅             ⋅             ⋅                ⋅            ⋅    ⋅ 

In [5]:
#Kφ = ∫(U ⋅ U * 0, Ω="body")
#Kφ[:,:]

In [6]:
mpc = MPC(master="remote", slave="right"; field=U, ux=true, uy=true)

#vagy egy mező esetén (nincs nyomaték)

mpc = MPC(master="remote", slave="right", uy=false)
bc2 = BoundaryCondition("remote", uy=0)

BoundaryCondition("remote", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0))

In [7]:
bc = BoundaryCondition("left", ux=0, uy=0)

fu = ∫(U ⋅ [1, 0], Γ="remote")
DoFs(fu);

In [8]:
u = solveField(Ku, fu, support=[bc, bc2], mpc=[mpc])

nodal VectorField
[0.0; 0.0; … ; 0.5999999999999999; 0.0;;]

In [9]:
showDoFResults(u)

0

In [10]:
bc3 = BoundaryCondition("remote", ux=0.1)

fu2 = ∫(U ⋅ [0, 0], Γ="remote")

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [11]:
u2 = solveField(Ku, fu2, support=[bc, bc2, bc3], mpc=[mpc])

nodal VectorField
[0.0; 0.0; … ; 0.1; 0.0;;]

In [12]:
showDoFResults(u2)

1

In [13]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
